In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score,recall_score,precision_score

In [ ]:
df = pd.read_csv("Crop_recommendation.csv")
df

In [ ]:
df.sample(20)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
crop_dict = {
    'rice': 1,
    'maize': 2,
    'jute': 3,
    'cotton': 4,
    'coconut': 5,
    'papaya': 6,
    'orange': 7,
    'apple': 8,
    'muskmelon': 9,
    'watermelon': 10,
    'grapes': 11,
    'mango': 12,
    'banana': 13,
    'pomegranate': 14,
    'lentil': 15,
    'blackgram': 16,
    'mungbean': 17,
    'mothbeans': 18,
    'pigeonpeas': 19,
    'kidneybeans': 20,
    'chickpea': 21,
    'coffee': 22
}
df['crop_num']=  df['label'].map(crop_dict)

In [ ]:
df.sample(20)

In [ ]:
X = df.drop(['label','crop_num'],axis=1)
y = df['crop_num']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

X_train

In [ ]:
y_train

In [ ]:
pipe_svm = Pipeline(
    steps=[

        ('scaler',StandardScaler()),
        ('model',SVC())
    ]
)
pipe_svm

pipe_rf = Pipeline(
    steps=[
        ('model',RandomForestClassifier())
    ]
)
pipe_rf



In [ ]:
grid_svm = [{
    "model__kernel" : ['linear'],
    "model__C" : [0.01,0.1,1,10,50,100]
},
      {
          "model__kernel" : ['rbf'],
          "model__C" : [0.1,1,10,50],
          "model__gamma" : [0.001,0.1,1,10,'scale','auto']
      } ,
      {
          "model__kernel" : ['poly'],
          "model__C" : [0.001,0.1,1,10,50,100],
          "model__degree" : [2,3]
      }
            ]


best_svm_model = GridSearchCV(
    estimator = pipe_svm,
    param_grid = grid_svm,
    cv = 5
)
best_svm_model.fit(X_train,y_train)



In [ ]:
svm_pred = best_svm_model.predict(X_test)

print(f"Accuracy Score : {accuracy_score(y_test,svm_pred)}")

In [ ]:
pipe_rf.fit(X_train,y_train)
rf_pred = pipe_rf.predict(X_test)

print(f"Accuracy score : {accuracy_score(y_test,rf_pred)}")

Prediction

In [ ]:
def recommendation(N,P,k,temperature,humidity,ph,rainfal):
    features = np.array([[N,P,k,temperature,humidity,ph,rainfal]])
    prediction = pipe_rf.predict(features)
    print(prediction)
    return prediction[0]

In [ ]:

N = 4
P = 134
k = 200
temperature =28.578288
humidity = 80.956290
ph = 5.840256
rainfall = 73.342321

predict = recommendation(N,P,k,temperature,humidity,ph,rainfall)

crop_dict = {1: "Rice", 2: "Maize", 3: "Jute", 4: "Cotton", 5: "Coconut", 6: "Papaya", 7: "Orange",
                 8: "Apple", 9: "Muskmelon", 10: "Watermelon", 11: "Grapes", 12: "Mango", 13: "Banana",
                 14: "Pomegranate", 15: "Lentil", 16: "Blackgram", 17: "Mungbean", 18: "Mothbeans",
                 19: "Pigeonpeas", 20: "Kidneybeans", 21: "Chickpea", 22: "Coffee"}

if predict in crop_dict:
    crop = crop_dict[predict]
    print("{} is a best crop to be cultivated ".format(crop))
else:
    print("Sorry are not able to recommend a proper crop for this environment")

In [ ]:
import pickle
pickle.dump(pipe_rf,open('model.pkl','wb'))